# large_pretrains.ipynb

Крупные предобученные модели для получения эмбеддингов ягод.
Продолжение `pretrains_check.ipynb` — здесь модели тяжелее, архитектуры сложнее.

| # | Модель | Размер | Особенность |
|---|--------|--------|-------------|
| 0 | Общая настройка | — | утилиты, данные |
| 1 | **LaBSE** (Google) | ~471 MB | кросс-лингвальные sentence-embeddings |
| 2 | **multilingual-E5-large** (Microsoft) | ~560 MB | query/passage-промпты |
| 3 | **BGE-M3** (BAAI) | ~570 MB | dense + sparse + ColBERT одновременно |
| 4 | **ai-forever/ru-en-RoSBERTa** (Сбер) | ~270 MB | RoBERTa, рус.+англ., SBERT-обучение |
| 5 | **XLM-RoBERTa-Large** (Meta) | ~1.1 GB | крупный multilingual baseline |
| 6 | **LoRA fine-tuning** XLM-R-Large | — | PEFT: дообучение без полного градиента |
| 7 | **Сравнение** всех моделей | — | PCA side-by-side + числовая таблица |

> **Память:** блоки 3 и 5 требуют 4–8 GB VRAM (или терпения на CPU).
> Каждый блок самодостаточен — запускай только то, что нужно.


## Блок 0 — Общая настройка

In [ ]:
import json, pathlib
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, "
          f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

pathlib.Path("pictures").mkdir(exist_ok=True)


In [ ]:
BERRIES = [
    "клубника", "земляника", "малина", "ежевика", "черника",
    "голубика", "брусника", "клюква", "крыжовник", "смородина", "облепиха",
]

# Описательные фразы — модели работают лучше, если дать контекст
BERRY_SENTENCES = [
    "клубника — красная садовая ягода",
    "земляника — маленькая красная лесная ягода",
    "малина — красная садовая ягода костянка",
    "ежевика — тёмная лесная ягода костянка",
    "черника — синяя лесная ягода",
    "голубика — синяя лесная ягода",
    "брусника — красная кислая лесная ягода",
    "клюква — красная кислая болотная ягода",
    "крыжовник — кислая садовая ягода на кусте",
    "смородина — садовая ягода на кусте",
    "облепиха — оранжевая кислая ягода на кусте",
]

# Для E5 и BGE-M3: формат «passage: <текст>» или «query: <текст>»
BERRY_PASSAGES = [f"passage: {s}" for s in BERRY_SENTENCES]
BERRY_QUERIES  = [f"query: {s}" for s in BERRY_SENTENCES]

# Загружаем корпус (нужен для блока 6)
try:
    with open("data/berry_corpus.json", encoding="utf-8") as f:
        corpus = json.load(f)
    print(f"Корпус загружен: {len(corpus)} doc, {sum(len(d) for d in corpus)} токенов")
except FileNotFoundError:
    corpus = None
    print("data/berry_corpus.json не найден — блок 6 (LoRA) будет пропущен")


In [ ]:
# ── Утилиты (те же, что в pretrains_check.ipynb) ──────────────────────────
def cosine_neighbors(query_vec, embs_dict, k=3):
    """k ближайших соседей по косинусу (исключая саму ягоду)."""
    labels = list(embs_dict.keys())
    matrix = np.stack(list(embs_dict.values()))
    q = query_vec / (np.linalg.norm(query_vec) + 1e-9)
    M = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9)
    sims = M @ q
    order = np.argsort(-sims)
    result = []
    for j in order:
        if labels[j] != None:
            result.append((labels[j], float(sims[j])))
        if len(result) == k + 1:
            break
    return result[1:]  # убираем саму ягоду (она на 0-м месте)

def show_neighbors(embs_dict, k=3):
    for berry, vec in embs_dict.items():
        nb = cosine_neighbors(vec, embs_dict, k=k)
        line = "  ".join(f"{n}({s:.2f})" for n, s in nb)
        print(f"  {berry:12s} → {line}")

def plot_pca(embs_dict, title, ax=None):
    labels = list(embs_dict.keys())
    matrix = np.stack(list(embs_dict.values()))
    xy = PCA(n_components=2).fit_transform(matrix)
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(xy[:, 0], xy[:, 1], s=45)
    for i, lbl in enumerate(labels):
        ax.annotate(lbl, (xy[i, 0], xy[i, 1]), fontsize=8,
                    xytext=(4, 3), textcoords="offset points")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    if own:
        plt.tight_layout(); plt.show()

# Пары, которые должны быть похожи — используем для числовой оценки
EXPECTED_PAIRS = [
    ("клубника",  "земляника"),
    ("малина",    "ежевика"),
    ("черника",   "голубика"),
    ("брусника",  "клюква"),
    ("крыжовник", "смородина"),
]

def score_pairs(embs_dict):
    sims = []
    for a, b in EXPECTED_PAIRS:
        if a in embs_dict and b in embs_dict:
            va = embs_dict[a] / (np.linalg.norm(embs_dict[a]) + 1e-9)
            vb = embs_dict[b] / (np.linalg.norm(embs_dict[b]) + 1e-9)
            sims.append(float(va @ vb))
    return float(np.mean(sims)) if sims else float("nan")

print("Утилиты готовы.")


---
## Блок 1 — LaBSE (Language-Agnostic BERT Sentence Embeddings, Google)

**Что это:** BERT, дообученный на 109 языках через dual-encoder с translation ranking loss.
Специально оптимизирован для того, чтобы переводы одного предложения давали близкие векторы.

- Архитектура: BERT-base (12 слоёв), dim=768
- Размер: ~471 MB
- Сильные стороны: кросс-лингвальность, хорошо работает на коротких фразах

**Использование:** через `sentence-transformers` как обычный SentenceTransformer.


In [ ]:
%pip install sentence-transformers --quiet


In [ ]:
from sentence_transformers import SentenceTransformer

labse_model = SentenceTransformer("sentence-transformers/LaBSE", device=str(device))
print(f"LaBSE загружен. Max seq len: {labse_model.max_seq_length}")


In [ ]:
# Вариант A: одно слово
labse_word_arr = labse_model.encode(BERRIES, normalize_embeddings=True, show_progress_bar=False)
labse_word = {b: labse_word_arr[i] for i, b in enumerate(BERRIES)}

# Вариант B: описательная фраза
labse_sent_arr = labse_model.encode(BERRY_SENTENCES, normalize_embeddings=True, show_progress_bar=False)
labse_sent = {b: labse_sent_arr[i] for i, b in enumerate(BERRIES)}

print(f"Dim = {labse_word_arr.shape[1]}")
print(f"Score (слово):  {score_pairs(labse_word):.4f}")
print(f"Score (фраза):  {score_pairs(labse_sent):.4f}")


In [ ]:
print("LaBSE (фраза) — ближайшие соседи:")
show_neighbors(labse_sent)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_pca(labse_word, "LaBSE (слово)", ax=axes[0])
plot_pca(labse_sent, "LaBSE (фраза)", ax=axes[1])
plt.suptitle("sentence-transformers/LaBSE", fontsize=13)
plt.tight_layout()
plt.savefig("pictures/labse_pca.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Блок 2 — multilingual-E5-large (Microsoft)

**Что это:** E5 (EmbEddings from bidirEctional Encoder rEpresentations) — модель, обученная
контрастивно на парах (query, passage) из веб-данных на 100 языках.

- Архитектура: XLM-RoBERTa-Large (24 слоя), dim=1024
- Размер: ~560 MB
- **Ключевое:** нужно добавлять префикс `"query: "` или `"passage: "` к тексту.
  Без префикса качество падает.

Инструкция от авторов:
- при поиске (запрос): `"query: <текст>"`
- при индексировании документов: `"passage: <текст>"`
- для симметричного сравнения: используй `"query: "` для обеих сторон


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

E5_MODEL = "intfloat/multilingual-e5-large"
e5_tok   = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model = AutoModel.from_pretrained(E5_MODEL).to(device).eval()

total_params = sum(p.numel() for p in e5_model.parameters())
print(f"multilingual-E5-large: {total_params/1e6:.0f}M параметров, dim=1024")


In [ ]:
@torch.no_grad()
def e5_encode(texts, tokenizer, model, batch_size=16, max_len=128):
    """
    Кодирование для E5-моделей.
    Тексты уже должны содержать префикс 'query: ' или 'passage: '.
    Используется mean-pooling + L2-нормализация (как в статье).
    """
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        # mean-pool с учётом attention mask
        mask = enc["attention_mask"].unsqueeze(-1).float()
        emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1)
        emb  = F.normalize(emb, p=2, dim=-1)
        all_embs.append(emb.cpu().float().numpy())
    return np.vstack(all_embs)


In [ ]:
# passage: — используем для «индексирования» ягод как документов
e5_pass_arr = e5_encode(BERRY_PASSAGES, e5_tok, e5_model)
e5_pass = {b: e5_pass_arr[i] for i, b in enumerate(BERRIES)}

# query: — используем для поиска (симметричный вариант тоже допустим)
e5_query_arr = e5_encode(BERRY_QUERIES, e5_tok, e5_model)
e5_query = {b: e5_query_arr[i] for i, b in enumerate(BERRIES)}

print(f"Dim = {e5_pass_arr.shape[1]}")
print(f"Score (passage): {score_pairs(e5_pass):.4f}")
print(f"Score (query):   {score_pairs(e5_query):.4f}")


In [ ]:
print("E5-large (passage:) — ближайшие соседи:")
show_neighbors(e5_pass)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_pca(e5_pass,  "E5-large (passage:)", ax=axes[0])
plot_pca(e5_query, "E5-large (query:)",   ax=axes[1])
plt.suptitle("intfloat/multilingual-e5-large", fontsize=13)
plt.tight_layout()
plt.savefig("pictures/e5_large_pca.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Блок 3 — BGE-M3 (BAAI)

**Что это:** state-of-the-art универсальная embedding-модель от Beijing Academy of AI.
Уникальная особенность — три режима retrieval в одной модели:

| Режим | Ключ | Как работает |
|-------|------|--------------|
| **Dense** | `dense_vecs` | обычный mean-pool, косинусное сходство |
| **Sparse (Lexical)** | `lexical_weights` | веса токенов → BM25-like scoring |
| **Multi-Vector (ColBERT)** | `colbert_vecs` | каждый токен = вектор, MaxSim |

- Архитектура: XLM-RoBERTa-Large (24 слоя), dim=1024
- Размер: ~570 MB
- Использует библиотеку `FlagEmbedding` (от авторов)

Для ягод особенно интересно сравнить dense vs. lexical —
sparse-метод может плохо работать на отдельных словах без совпадений токенов.


In [ ]:
%pip install FlagEmbedding --quiet


In [ ]:
from FlagEmbedding import BGEM3FlagModel

bge_model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=torch.cuda.is_available(),   # fp16 только если GPU
    device=str(device),
)
print("BGE-M3 загружен")


In [ ]:
# Кодируем ягоды — получаем все три типа векторов сразу
bge_output = bge_model.encode(
    BERRY_SENTENCES,
    batch_size=8,
    max_length=64,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
)

bge_dense   = {b: bge_output["dense_vecs"][i]   for i, b in enumerate(BERRIES)}
bge_lexical = {b: bge_output["lexical_weights"][i] for i, b in enumerate(BERRIES)}
bge_colbert = {b: bge_output["colbert_vecs"][i] for i, b in enumerate(BERRIES)}

print(f"Dense dim:   {bge_output['dense_vecs'].shape[1]}")
print(f"ColBERT dim: {bge_output['colbert_vecs'][0].shape}  (seq_len × hidden)")
print(f"Lexical:     dict из {{token_id: вес}} на каждую ягоду")


In [ ]:
# Для sparse (lexical) нам нужна специальная функция similarity
# BGE-M3 считает её через dot-product по sparse-словарям
def sparse_similarity(vec_a, vec_b):
    """Скалярное произведение разреженных весов токенов."""
    # vec_a / vec_b — dict {token_id: float}
    return sum(
        vec_a.get(k, 0.0) * v
        for k, v in vec_b.items()
    )

def colbert_maxsim(vecs_q, vecs_d):
    """MaxSim: для каждого токена запроса берём макс. косинус с токенами документа."""
    # vecs_q, vecs_d — numpy (seq_len, dim)
    q = vecs_q / (np.linalg.norm(vecs_q, axis=1, keepdims=True) + 1e-9)
    d = vecs_d / (np.linalg.norm(vecs_d, axis=1, keepdims=True) + 1e-9)
    sim_matrix = q @ d.T          # (seq_q, seq_d)
    return float(sim_matrix.max(axis=1).mean())  # среднее по токенам запроса


In [ ]:
# Сравниваем три режима для пары малина / ежевика
a, b = "малина", "ежевика"
print(f"Сходство {a} ↔ {b}:")

va_d = bge_dense[a]; vb_d = bge_dense[b]
print(f"  Dense   : {float(va_d @ vb_d):.4f}")

va_s = bge_lexical[a]; vb_s = bge_lexical[b]
print(f"  Sparse  : {sparse_similarity(va_s, vb_s):.4f}")

va_c = bge_colbert[a]; vb_c = bge_colbert[b]
print(f"  ColBERT : {colbert_maxsim(va_c, vb_c):.4f}")


In [ ]:
print("BGE-M3 Dense — соседи:")
show_neighbors(bge_dense)

print("\nScore пар:")
print(f"  Dense:  {score_pairs(bge_dense):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_pca(bge_dense, "BGE-M3 Dense (dim=1024)", ax=axes[0])

# Для ColBERT: усредним токены, чтобы получить один вектор для PCA
bge_colbert_mean = {b: v.mean(0) for b, v in bge_colbert.items()}
plot_pca(bge_colbert_mean, "BGE-M3 ColBERT (mean по токенам)", ax=axes[1])

plt.suptitle("BAAI/bge-m3", fontsize=13)
plt.tight_layout()
plt.savefig("pictures/bge_m3_pca.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Матрица сходства всех ягод (Dense)
labels = BERRIES
matrix = np.stack([bge_dense[b] for b in labels])
sim_matrix = matrix @ matrix.T

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0.3, vmax=1.0)
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, ax=ax)
ax.set_title("BGE-M3 Dense: матрица косинусного сходства")
plt.tight_layout()
plt.savefig("pictures/bge_m3_similarity_matrix.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Блок 4 — ai-forever/ru-en-RoSBERTa (Сбер AI)

**Что это:** RoBERTa-base, дообученная командой Сбер (AI Forever) на русско-английских
NLI-задачах по методологии Sentence-BERT (siamese-network, cosine similarity loss).

- Архитектура: RoBERTa-base (12 слоёв), dim=768
- Обучение: NLI + STS на рус.+англ. корпусах
- Размер: ~270 MB
- Плюс: лёгкая, хорошо откалибрована для симметричного сравнения фраз

Загружается через `sentence-transformers`.


In [ ]:
from sentence_transformers import SentenceTransformer

rosbert_model = SentenceTransformer("ai-forever/ru-en-RoSBERTa", device=str(device))
print(f"ru-en-RoSBERTa загружен. Dim: {rosbert_model.get_sentence_embedding_dimension()}")


In [ ]:
rosbert_word_arr = rosbert_model.encode(BERRIES,         normalize_embeddings=True, show_progress_bar=False)
rosbert_sent_arr = rosbert_model.encode(BERRY_SENTENCES, normalize_embeddings=True, show_progress_bar=False)

rosbert_word = {b: rosbert_word_arr[i] for i, b in enumerate(BERRIES)}
rosbert_sent = {b: rosbert_sent_arr[i] for i, b in enumerate(BERRIES)}

print(f"Score (слово): {score_pairs(rosbert_word):.4f}")
print(f"Score (фраза): {score_pairs(rosbert_sent):.4f}")


In [ ]:
print("ru-en-RoSBERTa (фраза) — соседи:")
show_neighbors(rosbert_sent)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_pca(rosbert_word, "RoSBERTa (слово)", ax=axes[0])
plot_pca(rosbert_sent, "RoSBERTa (фраза)", ax=axes[1])
plt.suptitle("ai-forever/ru-en-RoSBERTa", fontsize=13)
plt.tight_layout()
plt.savefig("pictures/rosbert_pca.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Блок 5 — XLM-RoBERTa-Large (Meta AI)

**Что это:** 100-языковая RoBERTa-Large, обученная на 2.5 TB CommonCrawl.
Это «фундаментальная» модель без специализации на embeddings —
эмбеддинги через mean-pool достаточно хороши, но E5/BGE/LaBSE обучались поверх неё специально.

- Архитектура: RoBERTa-Large (24 слоя, 16 голов), dim=1024
- Параметров: ~560M, размер на диске ~1.1 GB
- Полезно как **сильный baseline** и как **база для дообучения** (блок 6)

> На CPU этот блок работает медленно. Рекомендуется GPU.


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch, torch.nn.functional as F

XLMR_MODEL = "FacebookAI/xlm-roberta-large"
xlmr_tok   = AutoTokenizer.from_pretrained(XLMR_MODEL)
xlmr_model = AutoModel.from_pretrained(XLMR_MODEL).to(device).eval()

total = sum(p.numel() for p in xlmr_model.parameters())
print(f"XLM-RoBERTa-Large: {total/1e6:.0f}M параметров")


In [ ]:
@torch.no_grad()
def xlmr_encode(texts, tokenizer, model, batch_size=16, max_len=128):
    """Mean-pool + L2-нормализация (аналог E5, но без специального обучения)."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        mask = enc["attention_mask"].unsqueeze(-1).float()
        emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1)
        emb  = F.normalize(emb, p=2, dim=-1)
        all_embs.append(emb.cpu().float().numpy())
    return np.vstack(all_embs)


In [ ]:
xlmr_word_arr = xlmr_encode(BERRIES,         xlmr_tok, xlmr_model)
xlmr_sent_arr = xlmr_encode(BERRY_SENTENCES, xlmr_tok, xlmr_model)

xlmr_word = {b: xlmr_word_arr[i] for i, b in enumerate(BERRIES)}
xlmr_sent = {b: xlmr_sent_arr[i] for i, b in enumerate(BERRIES)}

print(f"Dim = {xlmr_word_arr.shape[1]}")
print(f"Score (слово): {score_pairs(xlmr_word):.4f}")
print(f"Score (фраза): {score_pairs(xlmr_sent):.4f}")


In [ ]:
print("XLM-RoBERTa-Large (фраза) — соседи:")
show_neighbors(xlmr_sent)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_pca(xlmr_word, "XLM-R-Large (слово)", ax=axes[0])
plot_pca(xlmr_sent, "XLM-R-Large (фраза)", ax=axes[1])
plt.suptitle("FacebookAI/xlm-roberta-large", fontsize=13)
plt.tight_layout()
plt.savefig("pictures/xlmr_large_pca.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Блок 6 — LoRA fine-tuning XLM-RoBERTa-Large на корпусе ягод

**Почему LoRA, а не полное дообучение?**

XLM-RoBERTa-Large имеет 560M параметров — full fine-tuning требует ~8–12 GB VRAM
только для хранения градиентов и оптимайзера. LoRA (Low-Rank Adaptation) решает это:

```
W_новый = W_исходный  +  B @ A     (rank=r << dim)
```

Обучаем только матрицы A, B (rank 8–16), это ~0.3% от весов модели.
Остальные веса заморожены. Результат — почти то же качество, в разы меньше памяти.

**Что делаем:** MLM на корпусе ягод → эмбеддинги должны стать более «ягодными».

> Требует `data/berry_corpus.json` из `embeddings_check.ipynb`.


In [ ]:
assert corpus is not None, "Нужен data/berry_corpus.json — запустите embeddings_check.ipynb"

%pip install peft --quiet


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForMaskedLM,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset as TorchDataset
import torch

# Используем xlm-roberta-large (уже знакомая нам модель)
LORA_BASE = "FacebookAI/xlm-roberta-large"
lora_tok   = AutoTokenizer.from_pretrained(LORA_BASE)
lora_base  = AutoModelForMaskedLM.from_pretrained(LORA_BASE)

total_orig = sum(p.numel() for p in lora_base.parameters())
print(f"Исходных параметров: {total_orig/1e6:.0f}M")


In [ ]:
# Применяем LoRA к Q и V проекциям attention-блоков
lora_config = LoraConfig(
    r=8,                            # ранг аппроксимации
    lora_alpha=16,                  # масштабирование (обычно 2*r)
    target_modules=["query", "value"],  # замораживаем всё, кроме Q и V
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.TOKEN_CLS,   # ближайший TaskType для MLM
)

lora_model = get_peft_model(lora_base, lora_config)
lora_model.print_trainable_parameters()


In [ ]:
class BerryMLMDataset(TorchDataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt",
        )
    def __len__(self):
        return self.enc["input_ids"].shape[0]
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.enc.items()}

flat_texts = [" ".join(doc) for doc in corpus]
lora_dataset = BerryMLMDataset(flat_texts, lora_tok)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=lora_tok,
    mlm=True,
    mlm_probability=0.15,
)

print(f"Датасет: {len(lora_dataset)} примеров")


In [ ]:
lora_args = TrainingArguments(
    output_dir="data/xlmr-large-berry-lora",
    num_train_epochs=5,
    per_device_train_batch_size=4,     # маленький батч для CPU
    gradient_accumulation_steps=4,     # эффективный батч = 16
    learning_rate=3e-4,                # LoRA обычно учат с большим lr
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
    no_cuda=not torch.cuda.is_available(),
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=lora_dataset,
    data_collator=data_collator,
)

print("Начинаем LoRA fine-tuning...")
trainer.train()
print("Готово!")


In [ ]:
# Сохраняем только LoRA-адаптеры (~несколько MB, не весь xlm-r-large)
lora_model.save_pretrained("data/xlmr-large-berry-lora")
lora_tok.save_pretrained("data/xlmr-large-berry-lora")
print("LoRA-адаптеры сохранены в data/xlmr-large-berry-lora/")
print("Загрузить снова: PeftModel.from_pretrained(base_model, 'data/xlmr-large-berry-lora')")


In [ ]:
# Извлекаем эмбеддинги из дообученной LoRA-модели
# (base encoder хранится в lora_model.base_model.model.roberta)
lora_model.eval().to(device)

@torch.no_grad()
def lora_encode(texts, tokenizer, model, batch_size=8, max_len=64):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        # У PeftModel для MLM базовый энкодер — через base_model
        out = model.base_model.model.roberta(**enc)
        mask = enc["attention_mask"].unsqueeze(-1).float()
        emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1)
        emb  = torch.nn.functional.normalize(emb, p=2, dim=-1)
        all_embs.append(emb.cpu().float().numpy())
    return np.vstack(all_embs)

lora_arr  = lora_encode(BERRY_SENTENCES, lora_tok, lora_model)
lora_embs = {b: lora_arr[i] for i, b in enumerate(BERRIES)}

print(f"Score до LoRA:   {score_pairs(xlmr_sent):.4f}")
print(f"Score после LoRA:{score_pairs(lora_embs):.4f}")


In [ ]:
print("XLM-R-Large + LoRA (фраза) — соседи:")
show_neighbors(lora_embs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_pca(xlmr_sent, "XLM-R-Large ДО LoRA", ax=axes[0])
plot_pca(lora_embs, "XLM-R-Large ПОСЛЕ LoRA", ax=axes[1])
plt.suptitle("LoRA fine-tuning XLM-RoBERTa-Large на корпусе ягод", fontsize=13)
plt.tight_layout()
plt.savefig("pictures/lora_finetuned_pca.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Блок 7 — Сравнение всех моделей

Собираем все запущенные блоки в одну сводку.


In [ ]:
# Регистрируем все доступные модели (используем вариант «фраза» как основной)
G = globals()
all_models = {}

if "labse_sent"    in G: all_models["LaBSE (фраза)"]              = G["labse_sent"]
if "e5_pass"       in G: all_models["E5-large (passage:)"]         = G["e5_pass"]
if "bge_dense"     in G: all_models["BGE-M3 Dense (фраза)"]        = G["bge_dense"]
if "rosbert_sent"  in G: all_models["ru-en-RoSBERTa (фраза)"]      = G["rosbert_sent"]
if "xlmr_sent"     in G: all_models["XLM-R-Large (фраза)"]         = G["xlmr_sent"]
if "lora_embs"     in G: all_models["XLM-R-Large + LoRA (фраза)"]  = G["lora_embs"]

print(f"Доступно моделей: {len(all_models)}")
for name in all_models:
    print(f"  • {name}")


In [ ]:
# PCA side-by-side
n = len(all_models)
if n == 0:
    print("Запусти хотя бы один блок выше.")
else:
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 7, rows * 6))
    axes_flat = np.array(axes).flatten()

    for i, (name, embs) in enumerate(all_models.items()):
        plot_pca(embs, name, ax=axes_flat[i])
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis("off")

    plt.suptitle("Сравнение крупных предобученных моделей — PCA 2D", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig("pictures/large_models_pca.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Сохранено: pictures/large_models_pca.png")


In [ ]:
# Сводная таблица: средняя косинусная близость «правильных пар»
print(f"{'Модель':<40} {'Score правильных пар':>22}  {'Dim':>6}")
print("-" * 72)

dim_map = {
    "LaBSE (фраза)": 768,
    "E5-large (passage:)": 1024,
    "BGE-M3 Dense (фраза)": 1024,
    "ru-en-RoSBERTa (фраза)": 768,
    "XLM-R-Large (фраза)": 1024,
    "XLM-R-Large + LoRA (фраза)": 1024,
}

for name, embs in all_models.items():
    sc  = score_pairs(embs)
    dim = dim_map.get(name, next(iter(embs.values())).shape[0])
    print(f"{name:<40} {sc:>22.4f}  {dim:>6}")


In [ ]:
# Тепловая карта попарного сходства между моделями
# (насколько «одинаково» разные модели видят ягодное пространство?)
if len(all_models) >= 2:
    model_names = list(all_models.keys())
    n_models = len(model_names)

    # Для каждой пары моделей — средний ранговый коэффициент Спирмена по всем ягодам
    from scipy.stats import spearmanr

    corr_matrix = np.eye(n_models)
    for i, name_i in enumerate(model_names):
        embs_i = all_models[name_i]
        for j, name_j in enumerate(model_names):
            if i >= j:
                continue
            embs_j = all_models[name_j]
            # Для каждой ягоды — ранги соседей
            berry_corrs = []
            for berry in BERRIES:
                if berry not in embs_i or berry not in embs_j:
                    continue
                # ранги всех ягод в пространстве i
                sims_i = np.array([
                    float(embs_i[berry] @ embs_i[b] /
                          (np.linalg.norm(embs_i[berry]) * np.linalg.norm(embs_i[b]) + 1e-9))
                    for b in BERRIES if b != berry
                ])
                sims_j = np.array([
                    float(embs_j[berry] @ embs_j[b] /
                          (np.linalg.norm(embs_j[berry]) * np.linalg.norm(embs_j[b]) + 1e-9))
                    for b in BERRIES if b != berry
                ])
                r, _ = spearmanr(sims_i, sims_j)
                berry_corrs.append(r)
            c = float(np.mean(berry_corrs))
            corr_matrix[i, j] = c
            corr_matrix[j, i] = c

    short_names = [n.split(" (")[0] for n in model_names]
    fig, ax = plt.subplots(figsize=(n_models * 1.5 + 1, n_models * 1.5 + 1))
    im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(n_models)); ax.set_yticks(range(n_models))
    ax.set_xticklabels(short_names, rotation=40, ha="right", fontsize=9)
    ax.set_yticklabels(short_names, fontsize=9)
    for i in range(n_models):
        for j in range(n_models):
            ax.text(j, i, f"{corr_matrix[i,j]:.2f}", ha="center", va="center", fontsize=9)
    plt.colorbar(im, ax=ax, label="Spearman ρ (ранги соседей)")
    ax.set_title("Согласованность моделей по порядку ягод-соседей", fontsize=11)
    plt.tight_layout()
    plt.savefig("pictures/large_models_agreement.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Сохранено: pictures/large_models_agreement.png")
